# 3. A complete inversion on each backend

A full-waveform inversion (FWI) starts from a model of the subsurface that knows little, simulates the survey
in it, compares the simulated seismograms with the recorded ones, and corrects the model with the gradient of
that misfit; again and again. Each correction costs a forward modelling and a gradient, so an inversion is
where the speed of the backend shows.

This notebook runs the same inversion on OpenCL and on CUDA, both in the `fixed` numeric mode (the same
mathematics), and compares the time, the way the misfit decreases, and the models found. The inversion is the
one of the [geophysical validation](../../docs/cuda-migration/reports/geophysical-validation-0.4.0.dev0.md) of the
fork: a faster disc in a uniform medium, sources and receivers at the surface, three frequency bands of 15 L-BFGS
iterations each. It takes about a minute and a half on a GeForce RTX 3060, almost all of it on OpenCL.

In [ ]:
import contextlib
import io
import time

import matplotlib.pyplot as plt
import numpy as np

from comparison import (  # the code shared by the notebooks of this folder: comparison.py
    CUDA_FIXED,
    OPENCL_FIXED,
    OPENCL_LEGACY,  # noqa: F401 (for ENGINES_TO_RUN below)
    check_engines,
    colour,
    inclusion_model,
    line_style,
    name,
    on_engine,
    propagator,
    style,
    survey,
)
from PyFWI.fwi import FWI

check_engines()

## Settings

The model (`NZ` × `NX` grid points 7 m apart), the sources (`NS`) and the length of the recording (`T`, s); the
frequency bands of the inversion, from low to high, and the iterations of each. `ENGINES_TO_RUN`: add
`OPENCL_LEGACY` to see what OpenCL gives by default (another numeric mode, so another result:
[notebook 2](2_same_results.ipynb)).

In [ ]:
NZ, NX, NS, T = 100, 100, 4, 0.6
FREQS = [15, 25, 45]  # Hz
ITERATIONS = [15, 15, 15]  # L-BFGS iterations of each frequency band
ENGINES_TO_RUN = [OPENCL_FIXED, CUDA_FIXED]

## The problem

The true model has vp = 3000 m/s in the disc and 2500 m/s around it. The inversion starts from 2500 m/s
everywhere and inverts vp only; vs and the density are the true ones, so that vp is the only unknown.

The source wavelet is 1000 times stronger than in the examples of PyFWI: the optimisers of the `legacy` mode stop
on absolute tolerances, which a unit source never exceeds (see `example/scripts/fwi.py`). It changes nothing in
the `fixed` mode, and lets `OPENCL_LEGACY` run too.

In [ ]:
problem = survey(NZ, NX, NS, T, amplitude=1000.0)
true = inclusion_model(NZ, NX)
start = {"vp": np.full((NZ, NX), 2500.0, np.float32), "vs": true["vs"], "rho": true["rho"]}

dh = problem["inpa"]["dh"]
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for ax, (title, model) in zip(axes, {"True vp": true, "Starting vp": start}.items()):
    image = ax.imshow(model["vp"], cmap="viridis", vmin=2400, vmax=3100)
    ax.plot(problem["rec_loc"][:, 0] / dh, problem["rec_loc"][:, 1] / dh, "w.", markersize=2, label="receivers")
    ax.plot(problem["src_loc"][:, 0] / dh, problem["src_loc"][:, 1] / dh, "rv", markersize=6, label="sources")
    ax.set_title(title, loc="left", fontsize=10)
    ax.set_xlabel("x (grid points)")
axes[0].set_ylabel("depth (grid points)")
axes[0].legend(loc="lower left", fontsize=8)
fig.colorbar(image, ax=axes, label="vp (m/s)")
plt.show()

## Recording the inversion

`FWI.fprime` is the method that runs one evaluation: a forward modelling, the misfit and the gradient. A subclass
records, at every evaluation, the time since the start, the misfit and the frequency band; nothing else changes.

In [ ]:
class TimedFWI(FWI):
    # FWI that records (seconds since the start, misfit, frequency band) at every evaluation.

    def __call__(self, *args, **kwargs):
        self.history, self.started = [], time.perf_counter()
        return super().__call__(*args, **kwargs)

    def fprime(self, m0, freq):
        misfit, gradient = super().fprime(m0, freq)
        self.history.append((time.perf_counter() - self.started, float(misfit), freq))
        return misfit, gradient

## Running the inversions

Each backend computes its own observed data from the true model, as a user of that backend would, then inverts
them. PyFWI prints two lines at every evaluation; they go to `logs` instead of the screen.

In [ ]:
runs, logs = {}, {}
for engine in ENGINES_TO_RUN:
    observed = propagator(problem, engine, chpr=0).forward_modeling(true, show=False)
    inpa = dict(on_engine(problem["inpa"], engine), energy_balancing=True)
    fwi = TimedFWI(observed, inpa, problem["src"], problem["rec_loc"], problem["shape"], components=0, chpr=20,
                   n_well_rec=problem["n_well_rec"])
    with contextlib.redirect_stdout(io.StringIO()) as logs[engine]:
        estimate, _ = fwi(start, method="lbfgs", freqs=FREQS, iter=ITERATIONS, n_params=1, k_0=1, k_end=2)
    seconds = time.perf_counter() - fwi.started
    runs[engine] = {"vp": estimate["vp"], "history": fwi.history, "seconds": seconds}
    print(f"{name(engine):15s} {seconds:6.1f} s, {len(fwi.history)} evaluations, "
          f"{seconds / len(fwi.history):.3f} s each")

first, *others = ENGINES_TO_RUN
for engine in others:
    print(f"{name(engine)} is {runs[first]['seconds'] / runs[engine]['seconds']:.1f} times faster than {name(first)}")

## How the misfit decreases

Left, the misfit after each evaluation: both backends follow the same path. Right, the same curves against the
clock: that is the difference. The misfit jumps up at the start of every frequency band, because each band
compares the data filtered up to another frequency.

In [ ]:
fig, (left, right) = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax in (left, right):
    style(ax)
for engine, run in runs.items():
    seconds, misfit, _ = zip(*run["history"])
    left.semilogy(np.arange(1, len(misfit) + 1), misfit, line_style(engine), color=colour(engine), label=name(engine))
    right.semilogy(seconds, misfit, line_style(engine), color=colour(engine), label=name(engine))
left.set_xlabel("evaluation (one forward modelling and one gradient)")
left.set_ylabel("misfit (log scale)")
right.set_xlabel("seconds since the start of the inversion")
left.legend(frameon=False)
left.set_title("Per evaluation: the same path", loc="left", fontsize=10)
right.set_title("Per second: the speed of the backend", loc="left", fontsize=10)
fig.tight_layout()
plt.show()

## The models found

How good is an estimate: the error ‖vp − true‖ / ‖start − true‖ (1: no progress, 0: the true model). How close the
two backends are: the norm of their difference over the norm of what the inversion changed in the model.

In [ ]:
def norm(a):
    return float(np.linalg.norm(np.asarray(a, np.float64)))


for engine, run in runs.items():
    print(f"{name(engine):15s} error {norm(run['vp'] - true['vp']) / norm(start['vp'] - true['vp']):.3f}")
for engine in others:
    difference = norm(runs[engine]["vp"] - runs[first]["vp"]) / norm(runs[first]["vp"] - start["vp"])
    print(f"{name(engine)} and {name(first)} differ by {100 * difference:.2f} % of what the inversion changed")

panels = {"True vp": true["vp"], **{name(engine): run["vp"] for engine, run in runs.items()}}
fig, axes = plt.subplots(1, len(panels) + 1, figsize=(4 * (len(panels) + 1), 3.8), sharey=True)
for ax, (title, vp) in zip(axes, panels.items()):
    image = ax.imshow(vp, cmap="viridis", vmin=2400, vmax=3100)
    ax.set_title(title, loc="left", fontsize=10)
    ax.set_xlabel("x (grid points)")
fig.colorbar(image, ax=axes[:-1], label="vp (m/s)", shrink=0.9)
difference = runs[others[0]]["vp"] - runs[first]["vp"] if others else np.zeros((NZ, NX))
limit = max(np.abs(difference).max(), 1e-3)
image = axes[-1].imshow(difference, cmap="RdBu_r", vmin=-limit, vmax=limit)
axes[-1].set_title(f"{name(others[0]) if others else ''} − {name(first)}", loc="left", fontsize=10)
axes[-1].set_xlabel("x (grid points)")
fig.colorbar(image, ax=axes[-1], label="difference (m/s)", shrink=0.9)
axes[0].set_ylabel("depth (grid points)")
plt.show()

## What this says

* **The same inversion runs several times faster on CUDA.** The geophysical validation measured 7.72 s on CUDA
  against 62.0 s on OpenCL for this inversion on the RTX 3060 of the project.
* **Both backends find the same model.** The misfit curves follow each other at first, then part a little:
  every evaluation rounds a few numbers differently, L-BFGS builds the next step on the previous ones, and these
  tiny differences grow from iteration to iteration, as they would on another computer. The models found differ
  by a small fraction of what the inversion recovers (printed above; at most 1.7 % over all the inversions of the
  geophysical validation, which also runs eight time-lapse strategies), and their errors are the same.
* The memory of the GPU, and the options that only CUDA has to save it, are in [notebook 5](5_memory.ipynb).